# 03 — BERTopic

BERTopic owns its own embedding (MiniLM) and clustering (UMAP + HDBSCAN
internally), so it's kept separate from notebook 02 rather than reusing
its cached embeddings.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import numpy as np
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

from utils import config
from utils.data import stratified_sample
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
texts = train_sample["text"].tolist()
true_labels = train_sample["label"].to_numpy()

In [3]:
umap_model = UMAP(n_neighbors=15, n_components=5, metric="cosine", random_state=config.SEED)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
# stop_words="english" so BERTopic's own c-TF-IDF representation surfaces
# meaningful topic words ("soccer", "goal", ...) instead of "the, to, of, in"
vectorizer_model = CountVectorizer(stop_words="english", min_df=2)
topic_model = BERTopic(embedding_model=sentence_model, umap_model=umap_model,
                        vectorizer_model=vectorizer_model,
                        nr_topics=config.NUM_CLASSES, calculate_probabilities=False)

topics, _ = topic_model.fit_transform(texts)
print(topic_model.get_topic_info())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   Topic  Count                      Name  \
0     -1   2849         -1_39_said_new_ap   
1      0   2392      0_39_new_reuters_oil   
2      1   1510          1_39_win_ap_game   
3      2   1249  2_iraq_said_president_39   

                                      Representation  \
0  [39, said, new, ap, reuters, lt, gt, quot, wor...   
1  [39, new, reuters, oil, gt, lt, said, prices, ...   
2  [39, win, ap, game, season, new, night, league...   
3  [iraq, said, president, 39, ap, bush, reuters,...   

                                 Representative_Docs  
0  [T-W, Comcast in Pact on Cable Unit Stake  NEW...  
1  [Avon Lowers U.S. Sales Forecast  CHICAGO (Reu...  
2  [Rallying Red Sox on Verge of Historic Win  NE...  
3  [Blair Coming to Washington for Discussions Wi...  


In [4]:
topics_arr = np.array(topics)
topic_embeddings = sentence_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
bertopic_metrics = evaluate_unsupervised(true_labels, topics_arr, topic_embeddings)
print(bertopic_metrics)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_bertopic.json", "w") as f:
    json.dump(bertopic_metrics, f, indent=2)
print("Saved BERTopic metrics.")

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

{'ACC (Hungarian)': np.float64(0.7165598912832459), 'NMI': 0.6605692501355472, 'ARI': 0.6126857294510963, 'FMI': 0.7391833003955273, 'Homogeneity': 0.5832207743444828, 'Completeness': 0.7615710737154081, 'V-Measure': 0.6605692501355473, 'Silhouette Score': 0.07383336871862411, 'Davies-Bouldin': 4.8054618184335185, 'Coverage': np.float64(0.643875)}
Saved BERTopic metrics.


### Topic inspection: what did each topic actually find?

BERTopic's `Representation` column above already gives per-topic c-TF-IDF
words (now stopword-filtered). This adds the same purity/majority-label
crosstab and example documents used in `02_unsupervised_clustering.ipynb`,
for a consistent qualitative view across all unsupervised methods —
eval-only, not used as a training signal.

In [5]:
summary = summarize_clusters(texts, topics_arr, true_labels, topic_embeddings, config.CLASS_NAMES)

if summary.empty:
    print("(no non-noise topics — everything was noise)")
else:
    with pd.option_context("display.max_colwidth", 60):
        print(summary.drop(columns="example_docs").to_string(index=False))
    for _, row in summary.iterrows():
        print(f"  topic {row['cluster']} examples:")
        for doc in row["example_docs"]:
            print(f"    - {doc}")

    summary.to_csv(config.RESULTS_DIR / "clusters_bertopic.csv", index=False)
    print(f"\nSaved per-topic qualitative summary to {config.RESULTS_DIR / 'clusters_bertopic.csv'}")

 cluster  size majority_true_label   purity                                                            top_terms
       0  2392            Sci/Tech 0.489967       39, new, reuters, oil, prices, gt, lt, microsoft, said, stocks
       1  1510              Sports 0.954967         39, ap, win, game, season, league, night, new, team, victory
       2  1249               World 0.862290 iraq, 39, president, said, ap, bush, baghdad, reuters, killed, iraqi
  topic 0 examples:
    - US Stocks Up Slightly After Ford Forecast  NEW YORK (Reuters) - U.S. blue chips edged higher on   Friday after Ford Moto
    - Stocks Open Lower; Intel, Oil Stocks Down  NEW YORK (Reuters) - U.S. stocks opened slightly lower on  Monday as chip mak
    - Intel Sets Upbeat Tone on Wall Street (Reuters) Reuters - U.S. shares are set to open higher on\Friday, buoyed by Intel 
  topic 1 examples:
    - A #39;s top M #39;s, maintain lead over Angels OAKLAND, Calif. - Bobby Crosbys sacrifice fly with one out in the ninth s
